<a href="https://colab.research.google.com/github/tmzt/TrainingExperiments/blob/main/Highbay/Local/HighbaySchemaProseFinetune3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Highbay schema/prose fine-tune - v3

Turns a plain-language automation request into the `GeniusAst` JSON the Genius
view consumes, and measures how often it gets the whole AST exactly right.

**Runs on a T4, and on anything better.** The dtype comes from compute
capability (fp16 on Turing, bf16 on Ampere+) and the batch size from VRAM, so an
A100 or L4 is not held to T4 settings. Nothing is hardcoded to one card.

## What v2 got wrong, and this fixes

**1. bf16 detection killed the trainer.** v2 (and v1) set the dtype from
`torch.cuda.is_bf16_supported()`. That helper defaults to
`including_emulation=True` and answers **True on a T4** (capability 7.5),
because bf16 *can* be emulated - while `TrainingArguments` checks compute
capability directly and raises *"Your setup doesn't support bf16/gpu ... You
need Ampere+ GPU"*. The two disagree and the helper's answer is the one that
reaches the trainer. v3 asks `get_device_capability()[0] >= 8`, in both the
runtime check and immediately before `TrainingArguments`.

**2. `trl<0.9.0` is older than Colab's transformers.** That pin came from v1.
Its `SFTTrainer` passes `tokenizer=` to `Trainer.__init__`, which current
transformers no longer accepts: `TypeError: Trainer.__init__() got an
unexpected keyword argument 'tokenizer'`. v3 uses `transformers.Trainer`
directly - no TRL API to track, and the response-only masking is written out
rather than borrowed, so it is inspectable.

**3. Nothing checked the length limits.** `max_new_tokens = 512` and
`MAX_SEQ_LENGTH = 2048` were magic numbers. v3 tokenizes the corpus and derives
both, then **asserts** no example is truncated - a truncated target is an AST
cut mid-JSON, which can never be matched and would just look like a bad model.
Generation is capped from the longest real target plus headroom, which also
roughly halves eval time.

## The data is already clean

`Highbay/Local/data/` in this repo is self-contained - raw sources, generator,
cleaned outputs - so this notebook clones and reads. No Drive staging for
input, no BOM handling, no merge, no normalization; `clean_genius_corpus.py`
did all of it, reproducibly from `source/`.

142 records, 71 `SCHEMA_SUGGESTION` / 71 `PIPELINE`, ~114 after the split.
Small for teaching a JSON schema from scratch: expect the model to lean on what
the base already knows, and read a large train/eval gap as a corpus problem
rather than a hyperparameter one. **Scope:** `Projects` and `Expenses` exist,
there is **no `Budget` table** across 100+ tables.

## Also carried from v2

The target is the **whole AST**, not `ui_prompt` (which is a *field* of the
AST, so targeting the AST yields the prose for free). The eval runs **before**
training as a baseline. Loss falls on the answer only. The adapter - not a
merged GGUF - is the artifact, and it goes to Drive.

In [ ]:
# 1. Runtime check - which dtype does this card actually support?
import subprocess, torch

print("GPU:", subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."

MAJOR, MINOR = torch.cuda.get_device_capability()

# ASK THE HARDWARE, not torch.cuda.is_bf16_supported(): that helper defaults to
# including_emulation=True and answers True on a T4, while TrainingArguments
# checks capability directly and refuses. Believing it is what made v2 die.
BF16 = MAJOR >= 8

print(f"compute capability {MAJOR}.{MINOR}   ->  bf16 (hardware) = {BF16}")
try:
    emulated = torch.cuda.is_bf16_supported()
    if emulated != BF16:
        print(f"  NOTE torch.cuda.is_bf16_supported() says {emulated} - it counts "
              "EMULATION and transformers does not. Never set bf16= from it.")
except Exception:
    pass
if not BF16:
    print("T4 path: fp16.")

# Size the run to the card. A T4 is 16 GB and Turing; an A100/L4 has both more
# memory and bf16, and should not be held to T4 batch settings.
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BATCH   = 2 if VRAM_GB < 24 else (4 if VRAM_GB < 48 else 8)
ACCUM   = max(1, 8 // BATCH)          # effective batch stays 8 either way
print(f"{VRAM_GB:.0f} GB VRAM  ->  per_device_batch {BATCH} x accum {ACCUM} "
      f"(effective {BATCH * ACCUM})")

In [ ]:
# 2. Dependencies.
# trl is UNPINNED: v1/v2 pinned trl<0.9.0, which is older than Colab's
# transformers and breaks with "Trainer.__init__() got an unexpected keyword
# argument 'tokenizer'". It is still installed because unsloth imports it, but
# this notebook does not use SFTTrainer.
!pip install torchao==0.18.0
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 3. Clone this repo and read the cleaned corpus. Public, so no auth, no Drive.
import os, json, random, collections

REPO_URL = "https://github.com/tmzt/TrainingExperiments.git"
CHECKOUT = "/content/TrainingExperiments"
DATA_DIR = f"{CHECKOUT}/Highbay/Local/data"

if not os.path.isdir(DATA_DIR):
    !git clone --depth 1 $REPO_URL $CHECKOUT

CORPUS = f"{DATA_DIR}/genius_corpus_clean.jsonl"
records = [json.loads(line) for line in open(CORPUS, encoding="utf-8") if line.strip()]

print(f"{len(records)} records from {CORPUS}")
print(dict(collections.Counter(r["output_ast"]["intent_type"] for r in records)))

In [ ]:
# 4. Contract. The eval needs it, and it guards against a bad regeneration
#    upstream - vocabularies read off the corpus, not invented.
INTENT_TYPES  = {"PIPELINE", "SCHEMA_SUGGESTION"}
TRIGGER_TYPES = {"ON_CREATE", "ON_UPDATE", "ON_DELETE", "SCHEDULED"}
ACTION_TYPES  = {"SEND_NOTIFICATION", "UPDATE_RECORD", "CALCULATE"}
COLUMN_TYPES  = {"NUMBER", "DATE", "STRING", "BOOLEAN", "RELATION"}
# TWO mutation shapes. Reading m["table"] skips every table creation, quietly.
MUTATION_KEYS = {
    "CREATE_TABLE": {"action", "name"},
    "ADD_COLUMN":   {"action", "table", "column_name", "type"},
}

def canonical(ast):
    """ONE string form per AST, so the training target and the eval comparison
    are the same definition rather than two that drift."""
    return json.dumps(ast, sort_keys=True, separators=(",", ":"))

def validate(ast):
    problems = []
    if not isinstance(ast, dict):
        return ["not an object"]
    intent = ast.get("intent_type")
    if intent not in INTENT_TYPES:
        problems.append(f"intent_type={intent!r}")
    if intent == "PIPELINE":
        p = ast.get("pipeline_ast")
        if not isinstance(p, dict):
            problems.append("PIPELINE without pipeline_ast")
        else:
            trig, act = p.get("trigger"), p.get("action")
            if not isinstance(trig, dict): problems.append("trigger missing")
            elif trig.get("type") not in TRIGGER_TYPES:
                problems.append(f"trigger.type={trig.get('type')!r}")
            if not isinstance(act, dict): problems.append("action missing")
            elif act.get("type") not in ACTION_TYPES:
                problems.append(f"action.type={act.get('type')!r}")
    if intent == "SCHEMA_SUGGESTION":
        muts = ast.get("schema_mutations")
        if not isinstance(muts, list) or not muts:
            problems.append("SCHEMA_SUGGESTION without schema_mutations")
        else:
            for i, m in enumerate(muts):
                want = MUTATION_KEYS.get(m.get("action"))
                if want is None:
                    problems.append(f"mutation[{i}].action={m.get('action')!r}")
                elif set(m) != want:
                    problems.append(f"mutation[{i}] keys {sorted(set(m))} != {sorted(want)}")
                elif m["action"] == "ADD_COLUMN" and m.get("type") not in COLUMN_TYPES:
                    problems.append(f"mutation[{i}].type={m.get('type')!r}")
    return problems

bad = [(i, p) for i, p in ((i, validate(r["output_ast"])) for i, r in enumerate(records)) if p]
print(f"{len(records) - len(bad)}/{len(records)} well-formed")
for i, p in bad[:8]:
    print("  record", i, p)
assert not bad, "the committed corpus does not validate - regenerate it"

In [ ]:
# 5. Model: 4-bit Llama-3.2-3B-Instruct + LoRA
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 2048          # checked against the real corpus in cell 7

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# The adapter is the only artifact this run produces, so its size is worth
# seeing: trainable params x 2 bytes is roughly what lands on disk.
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"adapter will be about {trainable * 2 / 1024**2:.0f} MiB at fp16, "
      "against ~2 GB for a merged q4_k_m GGUF")

# No mapping= : that argument maps ShareGPT-shaped data, and these are
# role/content messages built here, so in v1 it applied to nothing.
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

In [ ]:
# 6. ChatML, and a stratified split with a fixed seed.
TARGET = "ast"          # "ast" (default) | "ui_prompt" (v1's behaviour)

SYSTEM = (
    "You convert a user's plain-language automation request into a strict JSON "
    "AST. Reply with JSON only - no prose, no code fences."
)
SYSTEM_PROSE = (
    "You restate a user's plain-language automation request as a short "
    "confirming question. Reply with one sentence."
)

def system_for():
    return SYSTEM_PROSE if TARGET == "ui_prompt" else SYSTEM

def answer_for(record):
    if TARGET == "ui_prompt":
        return record["output_ast"].get("ui_prompt", "")
    return canonical(record["output_ast"])

def to_chatml(record):
    return {"messages": [
        {"role": "system",    "content": system_for()},
        {"role": "user",      "content": record["user_input"]},
        {"role": "assistant", "content": answer_for(record)},
    ]}

def to_text(record):
    return tokenizer.apply_chat_template(
        to_chatml(record)["messages"], tokenize = False, add_generation_prompt = False)

if TARGET == "ast":
    recovered = [json.loads(to_chatml(r)["messages"][2]["content"]) for r in records]
    assert all(canonical(a) == canonical(r["output_ast"])
               for a, r in zip(recovered, records)), "an AST changed through ChatML"
    print(f"all {len(records)} ASTs recovered byte-identical from the assistant turn")

SEED, EVAL_FRACTION = 3407, 0.2
by_intent = collections.defaultdict(list)
for r in records:
    by_intent[r["output_ast"].get("intent_type")].append(r)

train_recs, eval_recs = [], []
rng = random.Random(SEED)
for intent, group in sorted(by_intent.items()):
    group = group[:]; rng.shuffle(group)
    cut = max(1, round(len(group) * EVAL_FRACTION))
    eval_recs += group[:cut]; train_recs += group[cut:]
rng.shuffle(train_recs); rng.shuffle(eval_recs)
print(f"train {len(train_recs)}  eval {len(eval_recs)}")

## 7. Lengths, measured rather than guessed

v2 carried `max_new_tokens = 512` and `MAX_SEQ_LENGTH = 2048` as magic numbers
and checked neither. Both matter:

* **A truncated training example** is an AST cut mid-JSON. It can never be
  matched, and the damage shows up as a bad model rather than as an error - so
  this asserts instead of warning.
* **`max_new_tokens` too low** truncates generation and every eval is a miss
  for a reason that has nothing to do with the model.
* **`max_new_tokens` too high** just wastes eval time, and there are ~68
  generations per run.

In [ ]:
# 7. Derive the limits from the corpus.
full_lens   = [len(tokenizer(to_text(r), add_special_tokens=False)["input_ids"])
               for r in records]
target_lens = [len(tokenizer(answer_for(r), add_special_tokens=False)["input_ids"])
               for r in records]

LONGEST_FULL, LONGEST_TARGET = max(full_lens), max(target_lens)
print(f"full example tokens : median {sorted(full_lens)[len(full_lens)//2]}  max {LONGEST_FULL}")
print(f"target tokens       : median {sorted(target_lens)[len(target_lens)//2]}  max {LONGEST_TARGET}")

# A truncated target can never be matched - fail here, not in the metric.
assert LONGEST_FULL <= MAX_SEQ_LENGTH, (
    f"longest example is {LONGEST_FULL} tokens but MAX_SEQ_LENGTH is "
    f"{MAX_SEQ_LENGTH}: raise it and reload the model, or targets get cut "
    "mid-JSON and can never match")

# Headroom for an untuned model that rambles before closing the JSON, without
# paying 512 tokens on every one of ~68 eval generations.
MAX_NEW_TOKENS = int(LONGEST_TARGET * 1.5) + 32
print(f"\nMAX_SEQ_LENGTH {MAX_SEQ_LENGTH} (headroom {MAX_SEQ_LENGTH - LONGEST_FULL} tokens)")
print(f"MAX_NEW_TOKENS {MAX_NEW_TOKENS} (1.5x the longest target + 32)")

## 8. Baseline, before any training

Without this number the post-training one cannot be read: a 3B instruct model
already emits plausible JSON.

In [ ]:
# 8. Eval: exact match on the canonical AST
def prompt_for(user_input):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system_for()},
         {"role": "user",   "content": user_input}],
        tokenize = False, add_generation_prompt = True)

@torch.no_grad()
def predict(user_input, max_new_tokens = None):
    ids = tokenizer(prompt_for(user_input), return_tensors = "pt",
                    add_special_tokens = False).to(model.device)
    out = model.generate(
        **ids, max_new_tokens = max_new_tokens or MAX_NEW_TOKENS, do_sample = False,
        pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:],
                            skip_special_tokens = True).strip()

def parse_ast(text):
    """Models like to wrap JSON in fences or trail prose."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        text = text[4:] if text.lower().startswith("json") else text
    start = text.find("{")
    if start < 0: return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        depth += (ch == "{") - (ch == "}")
        if depth == 0:
            try: return json.loads(text[start:i + 1])
            except json.JSONDecodeError: return None
    return None      # never closed: ran out of tokens, or never was JSON

def evaluate(dataset, label):
    if TARGET != "ast":
        print(f"--- {label}: TARGET={TARGET!r}, exact-AST does not apply ---")
        for r in dataset[:3]:
            print("  in :", r["user_input"][:70])
            print("  out:", predict(r["user_input"], 128)[:120])
        return 0.0
    n = len(dataset); parsed = exact = valid = intent_ok = unclosed = 0
    misses = []
    for r in dataset:
        raw = predict(r["user_input"])
        got, want = parse_ast(raw), r["output_ast"]
        if got is None:
            # Distinguish "ran out of room" from "not JSON at all" - the first
            # is a MAX_NEW_TOKENS problem, not a model problem.
            if raw.count("{") > raw.count("}"): unclosed += 1
            misses.append((r["user_input"], f"unparseable: {raw[:80]!r}")); continue
        parsed += 1
        if not validate(got): valid += 1
        if canonical(got) == canonical(want): exact += 1
        else: misses.append((r["user_input"], canonical(got)[:140]))
        intent_ok += got.get("intent_type") == want.get("intent_type")
    print(f"--- {label}  (n={n}) ---")
    print(f"  parseable JSON  {parsed}/{n}")
    print(f"  schema-valid    {valid}/{n}")
    print(f"  EXACT AST       {exact}/{n}   <- the number that matters")
    print(f"  intent_type     {intent_ok}/{n}")
    if unclosed:
        print(f"  !! {unclosed} outputs had unclosed JSON - raise MAX_NEW_TOKENS "
              f"(currently {MAX_NEW_TOKENS}); those are not model errors")
    for inp, got in misses[:5]:
        print(f"    miss {inp[:55]!r}\n         -> {got}")
    return exact / n if n else 0.0

FastLanguageModel.for_inference(model)
baseline = evaluate(eval_recs, "BASELINE (no fine-tuning)")

In [ ]:
# 9. Train - plain transformers.Trainer.
# NOT trl.SFTTrainer: the trl v1 pinned passes tokenizer= to Trainer.__init__,
# which current transformers rejects. Trainer has no such coupling, and writing
# the masking out makes it inspectable rather than borrowed.
from transformers import Trainer, TrainingArguments
from datasets import Dataset

FastLanguageModel.for_training(model)
model.config.use_cache = False

# Recompute from the hardware: if cell 1 was not re-run on this runtime, a stale
# BF16 would reach TrainingArguments and raise "You need Ampere+ GPU".
BF16 = torch.cuda.get_device_capability()[0] >= 8
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BATCH   = 2 if VRAM_GB < 24 else (4 if VRAM_GB < 48 else 8)
ACCUM   = max(1, 8 // BATCH)
print(f"training in {'bf16' if BF16 else 'fp16'} on {VRAM_GB:.0f} GB "
      f"(capability {torch.cuda.get_device_capability()}), "
      f"batch {BATCH} x accum {ACCUM}")

# Loss on the ANSWER only - everything up to and including the assistant header
# is masked to -100. Otherwise most of the gradient goes on reproducing the
# question, which at ~114 records is most of the signal.
ASSIST = tokenizer("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]

def encode(record):
    ids = tokenizer(to_text(record), truncation=True, max_length=MAX_SEQ_LENGTH,
                    add_special_tokens=False)["input_ids"]
    cut = 0
    for i in range(len(ids) - len(ASSIST), -1, -1):        # last occurrence
        if ids[i:i + len(ASSIST)] == ASSIST:
            cut = i + len(ASSIST); break
    return {"input_ids": ids, "labels": [-100] * cut + ids[cut:]}

train_ds = Dataset.from_list([encode(r) for r in train_recs])

scored = sum(l != -100 for l in train_ds[0]["labels"])
total  = len(train_ds[0]["input_ids"])
print(f"example: {total} tokens, {scored} scored ({scored/total:.0%})")
assert 0 < scored < total, (
    "masking failed - the assistant header did not match, so this would train "
    "on the prompt as well as the answer")

pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    return {
        "input_ids": torch.tensor(
            [b["input_ids"] + [pad_id] * (width - len(b["input_ids"])) for b in batch]),
        "labels": torch.tensor(
            [b["labels"] + [-100] * (width - len(b["labels"])) for b in batch]),
        "attention_mask": torch.tensor(
            [[1] * len(b["input_ids"]) + [0] * (width - len(b["input_ids"])) for b in batch]),
    }

trainer = Trainer(
    model = model,
    train_dataset = train_ds,
    data_collator = collate,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH,     # from VRAM, cell 1
        gradient_accumulation_steps = ACCUM,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not BF16,          # T4 lands here
        bf16 = BF16,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = SEED,
        output_dir = "outputs",
        report_to = [],
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 10. Eval after training, against the baseline
FastLanguageModel.for_inference(model)
model.config.use_cache = True          # generation is slow without it

tuned = evaluate(eval_recs, "AFTER FINE-TUNING")
if TARGET == "ast":
    print(f"\nexact-AST  baseline {baseline:.1%}  ->  tuned {tuned:.1%}"
          f"   (delta {tuned - baseline:+.1%})")
    _ = evaluate(train_recs[:10], "TRAIN SUBSET (memorization check)")
    print("\nNear-perfect on train while eval lags is memorization, the expected "
          f"shape at {len(train_recs)} records. The fix is more corpus, not more "
          "epochs.")

## 11. The adapter is the artifact, and it goes to Drive

**You only need the upstream base plus the LoRA.** The base is fetched by id
and never stored; the adapter is the only thing this run produces. It is
written to `MyDrive/Training Data/genius_lora_v3`.

| artifact | size |
|---|---|
| LoRA adapter, fp16 | **~46 MiB** (24.3M params at r=16) |
| merged q4_k_m GGUF | ~2 GB |

GitHub's hard limit is 100 MiB per file, so the adapter would fit in this repo
and the GGUF would not (Git LFS does not rescue it either - 1 GB free tier).
Drive keeps run outputs out of git history entirely, which is the better reason.

A merged GGUF is only for a single standalone llama.cpp file, and even that is
avoidable: llama.cpp takes a GGUF LoRA adapter separately
(`convert_lora_to_gguf.py`, then `--lora`).

**Caveat whichever you pick:** this adapter is trained against a **4-bit** base,
so merging into a 16-bit base or applying it to a differently quantized one is
an approximation. Re-run the eval against whatever gets packaged.

Re-running **replaces** the Drive copy - the cell warns first.

In [ ]:
# 11. Save. The adapter goes to Drive; the GGUF is opt-in.
import shutil

SAVE_DIR = "/content/genius_lora_v3"
model.save_pretrained(SAVE_DIR)          # adapter only - base is fetched by id
tokenizer.save_pretrained(SAVE_DIR)

size = subprocess.run(["du", "-sh", SAVE_DIR], capture_output=True, text=True).stdout.split()[0]
print(f"adapter -> {SAVE_DIR}  ({size})")

SAVE_TO_DRIVE = True
DRIVE_DEST    = "/content/drive/MyDrive/Training Data/genius_lora_v3"
EXPORT_GGUF   = False
PUSH_TO_HF    = False
HF_REPO       = "tmzt/genius-schema-prose-v3"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.isdir(DRIVE_DEST):
        print(f"NOTE: {DRIVE_DEST} exists and is being REPLACED. "
              "Rename DRIVE_DEST first if that run is worth keeping.")
    os.makedirs(os.path.dirname(DRIVE_DEST), exist_ok=True)
    shutil.copytree(SAVE_DIR, DRIVE_DEST, dirs_exist_ok=True)
    print("adapter ->", DRIVE_DEST)
    print("  reload with FastLanguageModel.from_pretrained(DRIVE_DEST)")

if PUSH_TO_HF:
    from huggingface_hub import notebook_login
    notebook_login()
    model.push_to_hub(HF_REPO); tokenizer.push_to_hub(HF_REPO)

if EXPORT_GGUF:
    model.save_pretrained_gguf(f"{SAVE_DIR}_q4_k_m", tokenizer,
                               quantization_method = "q4_k_m")
    !du -sh {SAVE_DIR}_q4_k_m